In [ ]:
import sys
import os
import numpy as np
# Get the current directory of the notebook
notebook_dir = os.getcwd()

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
# Add the 2nd level parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(parent_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from inflow_model.blade_params import P600_Blade
from single_rotor_model import SingleRotorBemtModel
from single_rotor_objective import SingleRotorObjective
from fit_plotter import FitPlotter
from manager import FittingManager
from post_processing import make_lookup_table
from drone import parameters
import inflow_model.propeller_lookup_table as propeller_lookup_table


Prepare data for fitting.

In [ ]:
import data_factory

fitting_subfolder = "wind_free_space_cfd_single_propeller"
factory = data_factory.FittingFactory()
data_list = data_factory.generate_data_list(fitting_subfolder, '.csv')
print(f"Fitting Data list:")
for data in data_list:
    print(data)
datasets = factory.prepare_datasets(data_list)


Start fitting. Rotor 0 of P600 is clockwise (is_ccw_rotor0=False).

In [ ]:
is_multiseed = True
is_fine_tune = False
# init_guess = [7.5, 2.25, 2.25, np.radians(20)]  # cl_1, cl_2, cd, alpha_0
init_guess = None
manager = FittingManager.for_single_rotor(P600_Blade(), is_ccw_rotor0=False, datasets=datasets, init_guess=init_guess)
manager.run(is_multiseed=is_multiseed, is_fine_tune=is_fine_tune)


Generate lookup table from fitted parameters.

In [ ]:
# Replace with the fitted params from the run above
fitted_params = [7.5, 2.25, 2.25, np.radians(20)]  # cl_1, cl_2, cd, alpha_0
make_lookup_table(fitted_params, P600_Blade(), "p600_single_rotor", is_hover_only=False)


Plot BET-predicted vs measured force for rotor 0.

In [ ]:
# Replace with fitted params before plotting
fitted_params = [7.5, 2.25, 2.25, np.radians(20)]  # cl_1, cl_2, cd, alpha_0
model = SingleRotorBemtModel(P600_Blade(), is_ccw_rotor0=False)
model.blade.cl_1, model.blade.cl_2, model.blade.cd, model.blade.alpha_0 = fitted_params
model.bet_instance.refresh_blade()
model.adjust_resolution(is_fine_tune=True)
fig = FitPlotter.plot_single_rotor_fit(model, datasets[0], sample_step=5)
